In [0]:
# Databricks notebook: 10_top_merchants
# Purpose: Transform PhonePe top merchant JSON into a
# ranked state/district/pincode Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType,
    LongType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.top_merchants"

EXPECTED_COUNTRY_FILES = 34
EXPECTED_STATE_FILES = 1_178
EXPECTED_SOURCE_FILES = 1_212

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT TOP MERCHANT SOURCE FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

top_merchant_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "top") &
        (F.col("domain") == "merchant")
    )
    .alias("m")
)

top_merchant_source_df = (
    top_merchant_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

source_scope_summary = (
    top_merchant_source_df
    .groupBy("file_scope")
    .agg(
        F.countDistinct("source_path").alias("file_count")
    )
)

source_counts = {
    row["file_scope"]: row["file_count"]
    for row in source_scope_summary.collect()
}

actual_country_files = source_counts.get("country", 0)
actual_state_files = source_counts.get("state", 0)
actual_source_files = actual_country_files + actual_state_files

print("\nSource validation")
print(f"Country files: {actual_country_files:,}")
print(f"State files  : {actual_state_files:,}")
print(f"Total files  : {actual_source_files:,}")

assert actual_country_files == EXPECTED_COUNTRY_FILES
assert actual_state_files == EXPECTED_STATE_FILES
assert actual_source_files == EXPECTED_SOURCE_FILES

print("Top merchant source validation passed.")


# ============================================================
# 3. DEFINE EXPLICIT JSON SCHEMA
# ============================================================

top_merchant_record_schema = StructType([
    StructField("name", StringType(), True),
    StructField("entityName", StringType(), True),

    StructField("registeredCount", LongType(), True),

    # Included for possible future schema evolution.
    StructField("registeredMerchants", LongType(), True)
])

top_merchant_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "states",
                ArrayType(top_merchant_record_schema),
                True
            ),
            StructField(
                "districts",
                ArrayType(top_merchant_record_schema),
                True
            ),
            StructField(
                "pincodes",
                ArrayType(top_merchant_record_schema),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit top merchant schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_top_merchant_df = (
    top_merchant_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            top_merchant_schema
        )
    )
)

parse_summary = (
    parsed_top_merchant_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.states").isNull() &
                F.col("parsed_json.data.districts").isNull() &
                F.col("parsed_json.data.pincodes").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_top_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files          : {parse_summary['source_files']:,}")
print(
    f"Files without top data: "
    f"{parse_summary['files_without_top_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_top_data"] == 0, (
    "One or more files do not contain readable state, "
    "district, or pincode merchant rankings."
)

print("Top merchant JSON parsing passed.")


# ============================================================
# 5. FUNCTION TO EXTRACT A RANKING LEVEL
# ============================================================

def extract_merchant_ranking(
    source_df,
    array_name,
    geography_level
):
    return (
        source_df

        # Preserve the source array order as ranking position.
        .select(
            "*",
            F.posexplode(
                F.col(f"parsed_json.data.{array_name}")
            ).alias("rank_index", "rank_record")
        )

        .select(
            F.col("file_scope"),

            F.lit(geography_level)
                .alias("geography_level"),

            F.when(
                F.col("file_scope") == "state",
                F.col("state_slug")
            )
            .otherwise(F.lit(None).cast("string"))
            .alias("parent_state_slug"),

            F.coalesce(
                F.trim(F.col("rank_record.name")),
                F.trim(F.col("rank_record.entityName"))
            ).alias("entity_name"),

            F.lower(
                F.regexp_replace(
                    F.coalesce(
                        F.trim(F.col("rank_record.name")),
                        F.trim(F.col("rank_record.entityName"))
                    ),
                    "[^a-zA-Z0-9]+",
                    "-"
                )
            ).alias("entity_slug"),

            (F.col("rank_index") + F.lit(1))
                .cast("int")
                .alias("rank_position"),

            F.col("year").cast("int").alias("year"),
            F.col("quarter").cast("int").alias("quarter"),
            F.col("period"),

            F.coalesce(
                F.col("rank_record.registeredCount"),
                F.col("rank_record.registeredMerchants")
            )
            .cast("long")
            .alias("registered_merchant_count"),

            F.when(
                F.col("rank_record.registeredCount").isNotNull(),
                F.lit("registeredCount")
            )
            .when(
                F.col(
                    "rank_record.registeredMerchants"
                ).isNotNull(),
                F.lit("registeredMerchants")
            )
            .otherwise(F.lit(None))
            .alias("registered_count_source_field"),

            F.col("parsed_json.responseTimestamp")
                .alias("response_timestamp_epoch_ms"),

            F.col("relative_path"),
            F.col("source_path"),
            F.col("source_commit"),
            F.col("content_sha256"),
            F.col("ingested_at_utc")
        )
    )


# ============================================================
# 6. EXTRACT STATE, DISTRICT, AND PINCODE RANKINGS
# ============================================================

top_merchant_states_df = extract_merchant_ranking(
    parsed_top_merchant_df,
    "states",
    "state"
)

top_merchant_districts_df = extract_merchant_ranking(
    parsed_top_merchant_df,
    "districts",
    "district"
)

top_merchant_pincodes_df = extract_merchant_ranking(
    parsed_top_merchant_df,
    "pincodes",
    "pincode"
)

top_merchants_df = (
    top_merchant_states_df
    .unionByName(top_merchant_districts_df)
    .unionByName(top_merchant_pincodes_df)
)

print("\nState, district, and pincode merchant rankings extracted.")


# ============================================================
# 7. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    top_merchants_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("geography_level").isNull() |
                F.col("entity_name").isNull() |
                F.col("rank_position").isNull() |
                F.col("registered_merchant_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("registered_merchant_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_merchant_counts"),

        F.sum(
            F.when(
                F.col("rank_position") <= 0,
                1
            ).otherwise(0)
        ).alias("invalid_rank_positions"),

        F.countDistinct("source_path")
            .alias("represented_source_files"),

        F.min("registered_merchant_count")
            .alias("minimum_merchant_count"),

        F.max("registered_merchant_count")
            .alias("maximum_merchant_count")
    )
    .first()
)

duplicate_rank_keys = (
    top_merchants_df
    .groupBy(
        "source_path",
        "geography_level",
        "rank_position"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows         : {quality_result['total_rows']:,}")
print(
    f"Missing required values   : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative merchant counts  : "
    f"{quality_result['negative_merchant_counts']:,}"
)
print(
    f"Invalid ranking positions : "
    f"{quality_result['invalid_rank_positions']:,}"
)
print(
    f"Source files represented  : "
    f"{quality_result['represented_source_files']:,}"
)
print(
    f"Minimum merchant count    : "
    f"{quality_result['minimum_merchant_count']:,}"
)
print(
    f"Maximum merchant count    : "
    f"{quality_result['maximum_merchant_count']:,}"
)
print(f"Duplicate ranking keys    : {duplicate_rank_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_merchant_counts"] == 0
assert quality_result["invalid_rank_positions"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_rank_keys == 0

print("Top merchant quality checks passed.")


# ============================================================
# 8. DISPLAY COVERAGE
# ============================================================

print("\nRows by geography level and source field")

display(
    top_merchants_df
    .groupBy(
        "geography_level",
        "registered_count_source_field"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count")
    )
    .orderBy("geography_level")
)


# ============================================================
# 9. SAVE MANAGED DELTA TABLE
# ============================================================

(
    top_merchants_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 10. VALIDATE SAVED TABLE
# ============================================================

saved_top_merchants_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_top_merchants_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final top merchant Delta-table validation passed.")


# ============================================================
# 11. DISPLAY SAMPLE
# ============================================================

display(
    saved_top_merchants_df
    .select(
        "geography_level",
        "parent_state_slug",
        "entity_name",
        "rank_position",
        "year",
        "quarter",
        "period",
        "registered_merchant_count",
        "registered_count_source_field"
    )
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "rank_position"
    )
    .limit(30)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.top_merchants
Expected files: 1,212

Source validation
Country files: 34
State files  : 1,178
Total files  : 1,212
Top merchant source validation passed.

Explicit top merchant schema created.

JSON parsing
Source files          : 1,212
Files without top data: 0
Top merchant JSON parsing passed.

State, district, and pincode merchant rankings extracted.

Data-quality results
Total Silver rows         : 10,413
Missing required values   : 0
Negative merchant counts  : 0
Invalid ranking positions : 0
Source files represented  : 1,212
Minimum merchant count    : 1
Maximum merchant count    : 7,866,204
Duplicate ranking keys    : 0
Top merchant quality checks passed.

Rows by geography level and source field


geography_level,registered_count_source_field,row_count,source_file_count
district,registeredCount,10073,1212
state,registeredCount,340,34



Silver Delta table created: workspace.silver.top_merchants

Saved-table validation
Saved rows              : 10,413
Source files represented: 1,212
First period            : 2018-Q1
Last period             : 2026-Q2
Final top merchant Delta-table validation passed.


geography_level,parent_state_slug,entity_name,rank_position,year,quarter,period,registered_merchant_count,registered_count_source_field
district,null,bengaluru urban district,1,2018,1,2018-Q1,10750,registeredCount
district,null,mumbai suburban district,2,2018,1,2018-Q1,1102,registeredCount
district,null,thane district,3,2018,1,2018-Q1,306,registeredCount
district,null,bengaluru rural district,4,2018,1,2018-Q1,175,registeredCount
district,null,mumbai district,5,2018,1,2018-Q1,71,registeredCount
district,null,kolar district,6,2018,1,2018-Q1,69,registeredCount
district,null,pune district,7,2018,1,2018-Q1,62,registeredCount
district,null,palghar district,8,2018,1,2018-Q1,58,registeredCount
district,null,tumakuru district,9,2018,1,2018-Q1,49,registeredCount
district,null,bengaluru south district,10,2018,1,2018-Q1,27,registeredCount


# Silver Top Merchants — Completion Summary

The `workspace.silver.top_merchants` Delta table was successfully created and validated.

## Results

- Total ranked merchant records: **10,413**
- Source files represented: **1,212**
- Data coverage: **2018-Q1 to 2026-Q2**
- Missing required values: **0**
- Negative merchant counts: **0**
- Invalid ranking positions: **0**
- Duplicate ranking keys: **0**

## Table grain

Each record represents:

- One source geography
- One ranking geography level
- One ranking position
- One year and quarter

## Available data

- State and district merchant rankings
- Registered-merchant count
- Ranking position
- Original source-field name
- Complete source lineage

## Analytical use

This table supports:

- Merchant-adoption leaderboards
- Rank-movement analysis
- Regional acceptance-infrastructure comparisons
- Emerging merchant-market identification
- Ranking stability analysis
- Regional clustering and ML features